# RBF SVM Classifier for rest_int Classification

**Target:** `rest_int` (0: 1min, 1: 5min, 2: 10min rest)

**Method:**
- **RBF SVM** classifier via `SVC(kernel='rbf')`
- **PCA** for dimensionality reduction before SVM
- **StratifiedKFold** for hyperparameter tuning
- **Grid Search** over hyperparameters
- **MCCV:** 10 random train/test splits by user_id

In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, confusion_matrix

In [2]:
# Load data (use dataset_filtered_1_450.csv for full PSD; fallback to bands)
import os

if os.path.exists("dataset_filtered_1_450.csv"):
    df = pd.read_csv("dataset_filtered_1_450.csv")
    meta_cols = ["user_id", "rest_int", "exp", "seq"]
    feature_cols = [c for c in df.columns if c not in meta_cols]
    print(f"Loaded dataset_filtered_1_450.csv: {len(df)} samples, {len(feature_cols)} features")
else:
    df = pd.read_csv("dataset_filtered_bands.csv")
    meta_cols = ["user_id", "rest_int", "exp", "seq"]
    feature_cols = [c for c in df.columns if c not in meta_cols]
    print(f"Loaded dataset_filtered_bands.csv (fallback): {len(df)} samples, {len(feature_cols)} features")

unique_users = df["user_id"].unique()
n_users = len(unique_users)
n_train_users = int(n_users * 0.8)
n_test_users = n_users - n_train_users

print(f"MCCV split: {n_train_users} train users, {n_test_users} test users per run")

Loaded dataset_filtered_1_450.csv: 243 samples, 1350 features
MCCV split: 11 train users, 3 test users per run


In [3]:
# Pipeline: StandardScaler -> PCA -> RBF SVM
# PCA n_components capped by min(n_features, n_samples)
max_pc = min(len(feature_cols), len(df) - 1)
n_components_options = [n for n in [2, 3, 4, 5, 10, 15, 20] if n <= max_pc]
if not n_components_options:
    n_components_options = [min(10, max_pc)]

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(random_state=42)),
    ("svc", SVC(kernel="rbf", random_state=42)),
])

param_grid = {
    "pca__n_components": n_components_options,
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": ["scale", "auto"],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
print(f"Grid: n_components={n_components_options}, C=[0.1,1,10,100], gamma=['scale','auto']")

Grid: n_components=[2, 3, 4, 5, 10, 15, 20], C=[0.1,1,10,100], gamma=['scale','auto']


In [4]:
# MCCV: 10 random train/test splits by user_id
N_MCCV = 10
MCCV_SEEDS = list(range(42, 42 + N_MCCV))

mccv_f1 = []
mccv_acc = []
mccv_cm_list = []
best_params_list = []

for run, seed in enumerate(MCCV_SEEDS):
    np.random.seed(seed)
    shuffled_users = np.random.permutation(unique_users)
    train_users = shuffled_users[:n_train_users]
    test_users = shuffled_users[n_train_users:]

    train_mask = df["user_id"].isin(train_users)
    test_mask = df["user_id"].isin(test_users)
    df_train = df[train_mask]
    df_test = df[test_mask]

    X_train = df_train[feature_cols].values
    y_train = df_train["rest_int"].values
    X_test = df_test[feature_cols].values
    y_test = df_test["rest_int"].values

    grid_search = GridSearchCV(
        pipe, param_grid, cv=cv, scoring="f1_weighted", n_jobs=-1, verbose=0
    )
    grid_search.fit(X_train, y_train)

    y_pred = grid_search.best_estimator_.predict(X_test)
    f1 = f1_score(y_test, y_pred, average="weighted")
    acc = (y_pred == y_test).mean()
    cm = confusion_matrix(y_test, y_pred)

    mccv_f1.append(f1)
    mccv_acc.append(acc)
    mccv_cm_list.append(cm)
    best_params_list.append(grid_search.best_params_)

    print(f"Run {run+1}/{N_MCCV} (seed={seed}): F1={f1:.4f}, Acc={acc:.4f}  best={grid_search.best_params_}")

Run 1/10 (seed=42): F1=0.3155, Acc=0.3137  best={'pca__n_components': 10, 'svc__C': 100, 'svc__gamma': 'scale'}
Run 2/10 (seed=43): F1=0.2464, Acc=0.2593  best={'pca__n_components': 3, 'svc__C': 100, 'svc__gamma': 'scale'}
Run 3/10 (seed=44): F1=0.3089, Acc=0.3148  best={'pca__n_components': 3, 'svc__C': 100, 'svc__gamma': 'scale'}
Run 4/10 (seed=45): F1=0.3384, Acc=0.3519  best={'pca__n_components': 10, 'svc__C': 10, 'svc__gamma': 'scale'}
Run 5/10 (seed=46): F1=0.2736, Acc=0.3148  best={'pca__n_components': 4, 'svc__C': 100, 'svc__gamma': 'scale'}
Run 6/10 (seed=47): F1=0.3037, Acc=0.3137  best={'pca__n_components': 5, 'svc__C': 100, 'svc__gamma': 'scale'}
Run 7/10 (seed=48): F1=0.2356, Acc=0.3148  best={'pca__n_components': 4, 'svc__C': 100, 'svc__gamma': 'scale'}
Run 8/10 (seed=49): F1=0.3084, Acc=0.3333  best={'pca__n_components': 20, 'svc__C': 100, 'svc__gamma': 'scale'}
Run 9/10 (seed=50): F1=0.3295, Acc=0.3519  best={'pca__n_components': 3, 'svc__C': 100, 'svc__gamma': 'scale'}

In [5]:
# MCCV summary
print("--- MCCV Results (10 random train/test splits by user_id) ---")
print(f"Weighted F1: {np.mean(mccv_f1):.4f} ± {np.std(mccv_f1):.4f}")
print(f"Accuracy:     {np.mean(mccv_acc):.4f} ± {np.std(mccv_acc):.4f}")

mean_cm = np.mean(mccv_cm_list, axis=0).round(1)
print("\n--- Mean Confusion Matrix (across 10 runs) ---")
print(mean_cm)
print("(rows: 1min, 5min, 10min; cols: predicted)")

from collections import Counter
param_strs = [str(p) for p in best_params_list]
most_common = Counter(param_strs).most_common(1)[0]
print(f"\nMost frequent best params: {most_common[0]} (in {most_common[1]}/10 runs)")

--- MCCV Results (10 random train/test splits by user_id) ---
Weighted F1: 0.2869 ± 0.0412
Accuracy:     0.3202 ± 0.0250

--- Mean Confusion Matrix (across 10 runs) ---
[[7.9 4.9 4.6]
 [5.9 3.  8.5]
 [5.9 6.1 6. ]]
(rows: 1min, 5min, 10min; cols: predicted)

Most frequent best params: {'pca__n_components': 3, 'svc__C': 100, 'svc__gamma': 'scale'} (in 3/10 runs)
